In [13]:

from kazoo.client import KazooClient
import time
import random
import threading
from multiprocessing import Manager

zk_host = "127.0.0.1:2181"
zk = KazooClient(hosts=zk_host)
zk.start()
print(f"Status: {zk.state}")
zk.stop()

Status: CONNECTED


In [14]:
def run_animal(name, party_size):
    zk = KazooClient(hosts=zk_host)
    zk.start()
    root = "/zoo_barrier"
    path = f"{root}/{name}"
    zk.ensure_path(root)
    zk.create(path, ephemeral=True)

    while True:
        party = zk.get_children(root)
        if len(party) < party_size:
            time.sleep(1)
        else:
            break

    print(f"{name} entered the zoo")
    for i in range(5):
        time.sleep(0.2)
    
    zk.delete(path)
    zk.stop()
    print(f"{name} left")

party_size = 3
animals = ["Monkey", "Tiger", "Elephant"]
threads = []

for name in animals:
    t = threading.Thread(target=run_animal, args=(name, party_size))
    t.start()
    threads.append(t)

for t in threads:
    t.join()

Monkey entered the zoo
Tiger entered the zoo
Elephant entered the zoo
Monkey left
Tiger left
Elephant left


In [15]:
def run_philosopher(ph_id, total, counters, eat_limit):
    zk = KazooClient(hosts=zk_host)
    zk.start()
    root = "/philosophers"
    forks = f"{root}/forks"
    
    table_lock = zk.Lock(f"{root}/table", ph_id)
    left_fork = zk.Lock(f"{forks}/{ph_id}", ph_id)
    right_fork = zk.Lock(f"{forks}/{(ph_id + 1) % total}", ph_id)

    while counters[ph_id] < eat_limit:
        with table_lock:
            left_free = len(left_fork.contenders()) == 0
            right_free = len(right_fork.contenders()) == 0
            
            n_left = (ph_id - 1) % total
            n_right = (ph_id + 1) % total
            can_eat = counters[n_left] >= counters[ph_id] and counters[n_right] >= counters[ph_id]

            if left_free and right_free and can_eat:
                left_fork.acquire()
                right_fork.acquire()

        if left_fork.is_acquired and right_fork.is_acquired:
            print(f"Philosopher {ph_id}: eating")
            counters[ph_id] += 1
            time.sleep(0.5)
            left_fork.release()
            right_fork.release()
        else:
            time.sleep(0.1)

    zk.stop()

num_ph = 5
eat_limit = 3
with Manager() as manager:
    counters = manager.list([0] * num_ph)
    
    m_zk = KazooClient(hosts=zk_host)
    m_zk.start()
    for p in ["/philosophers", "/philosophers/table", "/philosophers/forks"]:
        if m_zk.exists(p): m_zk.delete(p, recursive=True)
        m_zk.ensure_path(p)
    m_zk.stop()

    threads = []
    for i in range(num_ph):
        t = threading.Thread(target=run_philosopher, args=(i, num_ph, counters, eat_limit))
        t.start()
        threads.append(t)

    for t in threads:
        t.join()
    print(f"Final counters: {list(counters)}")

Philosopher 0: eating
Philosopher 3: eating
Philosopher 2: eating
Philosopher 4: eating
Philosopher 1: eating
Philosopher 3: eating
Philosopher 0: eating
Philosopher 2: eating
Philosopher 4: eating
Philosopher 1: eating
Philosopher 0: eating
Philosopher 3: eating
Philosopher 2: eating
Philosopher 4: eating
Philosopher 1: eating
Final counters: [3, 3, 3, 3, 3]


In [ ]:
import threading
import random
import time
from kazoo.client import KazooClient
from kazoo.recipe.watchers import DataWatch, ChildrenWatch

VOTE_COMMIT = b'voted_commit'
VOTE_ABORT = b'voted_abort'
GLOBAL_COMMIT = b'global_commit'
GLOBAL_ABORT = b'global_abort'
COMMITTED = b'committed'

ZK_HOST = "127.0.0.1:2181"
TX_ROOT = "/app/tx"

class Participant(threading.Thread):
    def __init__(self, p_id, total_count):
        super().__init__()
        self.p_id = p_id
        self.path = f"{TX_ROOT}/node_{p_id}"
        self.zk = KazooClient(hosts=ZK_HOST)

    def run(self):
        self.zk.start()
        
        vote = VOTE_COMMIT if random.random() > 0.2 else VOTE_ABORT
        print(f"[Исполнитель {self.p_id}] Голосует: {vote.decode()}")
        
        self.zk.create(self.path, vote, ephemeral=True, makepath=True)

        @DataWatch(self.zk, self.path)
        def watch_decision(data, stat):
            if data in [GLOBAL_COMMIT, GLOBAL_ABORT]:
                decision = data.decode()
                print(f"[Исполнитель {self.p_id}] Получил решение: {decision}")
                
                time.sleep(0.1)
                
                if self.zk.exists(self.path):
                    self.zk.set(self.path, COMMITTED)
                    print(f"[Исполнитель {self.p_id}] Статус: {COMMITTED.decode()}")

        timeout = 10
        start_time = time.time()
        while time.time() - start_time < timeout:
            data, _ = self.zk.get(self.path)
            if data == COMMITTED:
                break
            time.sleep(0.5)
            
        self.zk.stop()
        self.zk.close()

class Coordinator(threading.Thread):
    def __init__(self, total_count):
        super().__init__()
        self.total_count = total_count
        self.zk = KazooClient(hosts=ZK_HOST)

    def run(self):
        self.zk.start()
        
        if self.zk.exists(TX_ROOT):
            self.zk.delete(TX_ROOT, recursive=True)
        self.zk.ensure_path(TX_ROOT)
        
        print(f"[Координатор] Ожидаю {self.total_count} исполнителей...")

        context = {"done": False}

        @ChildrenWatch(self.zk, TX_ROOT)
        def watch_children(children):
            if len(children) == self.total_count and not context["done"]:
                self.make_decision(children, context)

        for _ in range(20):
            if context["done"]: break
            time.sleep(0.5)
            
        self.zk.stop()
        self.zk.close()

    def make_decision(self, children, context):
        print("[Координатор] Все голоса собраны. Анализирую...")
        votes = []
        for c in children:
            data, _ = self.zk.get(f"{TX_ROOT}/{c}")
            votes.append(data)

        final_decision = GLOBAL_COMMIT if all(v == VOTE_COMMIT for v in votes) else GLOBAL_ABORT
        print(f"[Координатор] ИТОГОВОЕ РЕШЕНИЕ: {final_decision.decode()}")

        for c in children:
            path = f"{TX_ROOT}/{c}"
            if self.zk.exists(path):
                self.zk.set(path, final_decision)
        
        context["done"] = True

N = 3
coord = Coordinator(N)
coord.start()

time.sleep(1)

participants = [Participant(i, N) for i in range(N)]
for p in participants:
    p.start()

for p in participants:
    p.join()
coord.join()

[Координатор] Ожидаю 3 исполнителей...
[Исполнитель 0] Голосует: voted_commit
[Исполнитель 1] Голосует: voted_commit
[Исполнитель 2] Голосует: voted_commit
[Координатор] Все голоса собраны. Анализирую...
[Координатор] ИТОГОВОЕ РЕШЕНИЕ: global_commit
[Исполнитель 0] Получил решение: global_commit
[Исполнитель 1] Получил решение: global_commit
[Исполнитель 2] Получил решение: global_commit
[Исполнитель 0] Статус: committed
[Исполнитель 1] Статус: committed
[Исполнитель 2] Статус: committed
